# Paper Results Table - Compile All RQ Results

Run this after `00_setup_and_sanity.ipynb`, RQ1-RQ5, and the consolidated `RQ6_external_validation.ipynb`. It prints tables for the manuscript, a cautious discussion, and source links for reporting and XAI evaluation.

In [1]:
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
METRICS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"
FIG_DIR = NOTEBOOK_DIR / "outputs" / "figures"

print(f"Metrics directory: {METRICS_DIR}")
print(f"Figures directory: {FIG_DIR}")

Metrics directory: C:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\notebooks\outputs\metrics
Figures directory: C:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\notebooks\outputs\figures


In [2]:
def load_csv(filename, required=False):
    path = METRICS_DIR / filename
    if not path.exists():
        msg = f"MISSING: {filename}"
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return None
    df = pd.read_csv(path)
    print(f"Loaded {filename}: {len(df)} rows")
    return df


rq1 = load_csv("RQ1_cam_comparison.csv")
rq2 = load_csv("RQ2_faithfulness.csv")
rq3 = load_csv("RQ3_backbone_comparison.csv")
rq4 = load_csv("RQ4_agreement_results.csv")
rq6_perf = load_csv("RQ6_separate_performance.csv")
rq6_combined = load_csv("RQ6_combined_scenarios.csv")
rq6_xai = load_csv("RQ6_xai_summary.csv")
rq6_disc = load_csv("RQ6_disagreement_summary.csv")

available = {
    "RQ1": rq1 is not None,
    "RQ2": rq2 is not None,
    "RQ3": rq3 is not None,
    "RQ4": rq4 is not None,
    "RQ6 performance": rq6_perf is not None,
    "RQ6 XAI": rq6_xai is not None,
    "RQ6 agreement": rq6_disc is not None,
}
print("\nAvailability:")
for name, ok in available.items():
    print(f"  {name:<18} {'OK' if ok else 'MISSING'}")

Loaded RQ1_cam_comparison.csv: 500 rows
Loaded RQ2_faithfulness.csv: 100 rows
Loaded RQ3_backbone_comparison.csv: 600 rows
Loaded RQ4_agreement_results.csv: 150 rows
Loaded RQ6_separate_performance.csv: 5 rows
MISSING: RQ6_combined_scenarios.csv
Loaded RQ6_xai_summary.csv: 5 rows
Loaded RQ6_disagreement_summary.csv: 5 rows

Availability:
  RQ1                OK
  RQ2                OK
  RQ3                OK
  RQ4                OK
  RQ6 performance    OK
  RQ6 XAI            OK
  RQ6 agreement      OK


## Table 1 - RQ1 CAM Method Comparison

In [3]:
if rq1 is None:
    print("RQ1 data not available. Run RQ1_cam_variant_comparison.ipynb first.")
else:
    table1 = rq1.groupby("method").agg(
        n=("image_id", "count"),
        fap_05_mean=("fap_05", "mean"),
        fap_05_std=("fap_05", "std"),
        fap_03_mean=("fap_03", "mean"),
        entropy_mean=("entropy", "mean"),
        entropy_std=("entropy", "std"),
        confidence_mean=("confidence", "mean"),
    ).sort_values("fap_05_mean")
    print(table1.round(4))
    print("\nLaTeX:")
    print(table1.round(4).to_latex(float_format="%.4f"))

             n  fap_05_mean  fap_05_std  fap_03_mean  entropy_mean  \
method                                                               
GradCAM    125       0.0442      0.0453       0.0868        7.0243   
EigenCAM   125       0.0671      0.0292       0.1212        9.3640   
GradCAM++  125       0.0777      0.0382       0.1640       10.1709   
LayerCAM   125       0.0788      0.0372       0.1679       10.1857   

           entropy_std  confidence_mean  
method                                   
GradCAM         3.9510           0.9161  
EigenCAM        0.2396           0.9161  
GradCAM++       0.2162           0.9161  
LayerCAM        0.2191           0.9161  

LaTeX:
\begin{tabular}{lrrrrrrr}
\toprule
 & n & fap_05_mean & fap_05_std & fap_03_mean & entropy_mean & entropy_std & confidence_mean \\
method &  &  &  &  &  &  &  \\
\midrule
GradCAM & 125 & 0.0442 & 0.0453 & 0.0868 & 7.0243 & 3.9510 & 0.9161 \\
EigenCAM & 125 & 0.0671 & 0.0292 & 0.1212 & 9.3640 & 0.2396 & 0.9161 \\
GradC

## Table 2 - RQ2 Faithfulness

In [4]:
if rq2 is None:
    print("RQ2 data not available. Run RQ2_faithfulness.ipynb first.")
else:
    table2 = rq2.groupby("method").agg(
        n=("image_id", "count"),
        insertion_auc_mean=("insertion_auc", "mean"),
        insertion_auc_std=("insertion_auc", "std"),
        deletion_auc_mean=("deletion_auc", "mean"),
        deletion_auc_std=("deletion_auc", "std"),
        faithfulness_mean=("faithfulness", "mean"),
        faithfulness_std=("faithfulness", "std"),
        positive_faithfulness_rate=("faithfulness", lambda s: float((s > 0).mean())),
    ).sort_values("faithfulness_mean", ascending=False)
    print(table2.round(4))
    print("\nLaTeX:")
    print(table2.round(4).to_latex(float_format="%.4f"))

            n  insertion_auc_mean  insertion_auc_std  deletion_auc_mean  \
method                                                                    
GradCAM    50              0.2193             0.2120             0.1364   
GradCAM++  50              0.1851             0.2276             0.1610   

           deletion_auc_std  faithfulness_mean  faithfulness_std  \
method                                                             
GradCAM              0.1284             0.0828            0.1506   
GradCAM++            0.1269             0.0241            0.1472   

           positive_faithfulness_rate  
method                                 
GradCAM                          0.62  
GradCAM++                        0.40  

LaTeX:
\begin{tabular}{lrrrrrrrr}
\toprule
 & n & insertion_auc_mean & insertion_auc_std & deletion_auc_mean & deletion_auc_std & faithfulness_mean & faithfulness_std & positive_faithfulness_rate \\
method &  &  &  &  &  &  &  &  \\
\midrule
GradCAM & 50 & 0.2193 &

## Table 3 - RQ3 Backbone x CAM Method

In [5]:
if rq3 is None:
    print("RQ3 data not available. Run RQ3_backbone_xai_quality.ipynb first.")
else:
    table3 = rq3.groupby(["model", "cam_method"]).agg(
        n=("image_id", "count"),
        correct_rate=("correct", "mean"),
        confidence_mean=("confidence", "mean"),
        fap_05_mean=("fap_05", "mean"),
        entropy_mean=("entropy", "mean"),
    ).sort_values("fap_05_mean")
    print(table3.round(4))
    print("\nLaTeX:")
    print(table3.round(4).to_latex(float_format="%.4f"))

                              n  correct_rate  confidence_mean  fap_05_mean  \
model           cam_method                                                    
resnet50        GradCAM     100          0.84           0.8912       0.0458   
efficientnet_b2 GradCAM++   100          0.90           0.9498       0.0664   
resnet50        GradCAM++   100          0.84           0.8912       0.0735   
mobilenetv2_100 GradCAM++   100          0.84           0.9198       0.0963   
                GradCAM     100          0.84           0.9198       0.1061   
efficientnet_b2 GradCAM     100          0.90           0.9498       0.1781   

                            entropy_mean  
model           cam_method                
resnet50        GradCAM           7.2723  
efficientnet_b2 GradCAM++         8.2239  
resnet50        GradCAM++        10.1184  
mobilenetv2_100 GradCAM++         6.8454  
                GradCAM           7.1320  
efficientnet_b2 GradCAM           9.3330  

LaTeX:
\begin{tabular}

## Table 4 - RQ4 Inter-Method Agreement vs Correctness

In [6]:
if rq4 is None:
    print("RQ4 data not available. Run RQ4_agreement_vs_uncertainty.ipynb first.")
else:
    table4 = rq4.groupby("correct").agg(
        n=("image_id", "count"),
        confidence_mean=("confidence", "mean"),
        avg_jaccard_mean=("avg_jaccard", "mean"),
        avg_jaccard_std=("avg_jaccard", "std"),
        avg_spearman_mean=("avg_spearman", "mean"),
        min_jaccard_mean=("min_jaccard", "mean"),
    )
    table4.index = table4.index.map({0: "Incorrect", 1: "Correct"})
    print(table4.round(4))
    print("\nInterpretation note: this is exploratory if the incorrect group is small.")
    print("\nLaTeX:")
    print(table4.round(4).to_latex(float_format="%.4f"))

             n  confidence_mean  avg_jaccard_mean  avg_jaccard_std  \
correct                                                              
Incorrect   25           0.7584            0.5354           0.2166   
Correct    125           0.9161            0.2905           0.2048   

           avg_spearman_mean  min_jaccard_mean  
correct                                         
Incorrect             0.6706            0.3852  
Correct               0.2722            0.1041  

Interpretation note: this is exploratory if the incorrect group is small.

LaTeX:
\begin{tabular}{lrrrrrr}
\toprule
 & n & confidence_mean & avg_jaccard_mean & avg_jaccard_std & avg_spearman_mean & min_jaccard_mean \\
correct &  &  &  &  &  &  \\
\midrule
Incorrect & 25 & 0.7584 & 0.5354 & 0.2166 & 0.6706 & 0.3852 \\
Correct & 125 & 0.9161 & 0.2905 & 0.2048 & 0.2722 & 0.1041 \\
\bottomrule
\end{tabular}



## Table 5 - RQ6 External Validation Performance

In [7]:
if rq6_perf is None:
    print("RQ6 performance data not available. Run consolidated RQ6_external_validation.ipynb first.")
else:
    cols = [
        "dataset", "n", "auc", "accuracy", "mean_confidence", "brier_score",
        "auc_drop_from_ham10000", "benign_accuracy", "malignant_accuracy",
    ]
    table5 = rq6_perf[[c for c in cols if c in rq6_perf.columns]].copy()
    print(table5.round(4).to_string(index=False))
    print("\nLaTeX:")
    print(table5.round(4).to_latex(index=False, float_format="%.4f"))

         dataset     n    auc  accuracy  mean_confidence  brier_score  auc_drop_from_ham10000  benign_accuracy  malignant_accuracy
        HAM10000  9815 0.9479    0.8699           0.8963       0.0917                  0.0000           0.8676              0.8815
        ISIC2020 32926 0.6348    0.9126           0.8936       0.0704                  0.3131           0.9245              0.2483
         MILK10K 10034 0.6677    0.4202           0.8625       0.4598                  0.2802           0.9219              0.2154
Malignant-Benign 34793 0.7799    0.6489           0.8655       0.2671                  0.1681           0.9268              0.3708
             PH2   200 0.7450    0.5150           0.9266       0.4217                  0.2029           0.9875              0.2000

LaTeX:
\begin{tabular}{lrrrrrrrr}
\toprule
dataset & n & auc & accuracy & mean_confidence & brier_score & auc_drop_from_ham10000 & benign_accuracy & malignant_accuracy \\
\midrule
HAM10000 & 9815 & 0.9479 & 0.8699

## Table 6 - RQ6 Combined Scenario Performance (Optional Extended Notebook)

In [8]:
if rq6_combined is None or len(rq6_combined) == 0:
    print("Optional RQ6 combined scenario data not available. This is expected unless you also run RQ6_extended_datasets.ipynb or export RQ6_combined_scenarios.csv.")
else:
    cols = [
        "scenario", "datasets", "n", "auc", "accuracy", "mean_confidence", "brier_score",
        "auc_drop_from_ham10000", "benign_accuracy", "malignant_accuracy",
    ]
    table6 = rq6_combined[[c for c in cols if c in rq6_combined.columns]].copy()
    print(table6.round(4).to_string(index=False))
    print("\nLaTeX:")
    print(table6.round(4).to_latex(index=False, float_format="%.4f"))

Optional RQ6 combined scenario data not available. This is expected unless you also run RQ6_extended_datasets.ipynb or export RQ6_combined_scenarios.csv.


## Table 7 - RQ6 XAI Robustness and Zero-CAM Failures

In [9]:
if rq6_xai is None:
    print("RQ6 XAI summary not available. Run consolidated RQ6_external_validation.ipynb first.")
else:
    table7 = rq6_xai.copy()
    print(table7.round(4).to_string(index=False))
    print("\nNote: FAP and entropy means exclude zero-CAM rows. Zero CAMs are reported as explanation failures.")
    print("\nLaTeX:")
    print(table7.round(4).to_latex(index=False, float_format="%.4f"))

         dataset  n_xai  zero_cam_count  n_valid  fap_05_mean  entropy_mean  confidence_mean  correct_rate  zero_cam_rate
        HAM10000   9785            9274      511       0.0573        9.2627           0.8690        0.8454         0.9478
        ISIC2020  32896           31026     1870       0.0517        9.3139           0.8799        0.9011         0.9432
         MILK10K  10004            9384      620       0.0533        9.3952           0.8582        0.4032         0.9380
Malignant-Benign  34763           32749     2014       0.0573        9.4743           0.8504        0.6137         0.9421
             PH2    170             159       11       0.0806        9.7395           0.9087        0.7273         0.9353

Note: FAP and entropy means exclude zero-CAM rows. Zero CAMs are reported as explanation failures.

LaTeX:
\begin{tabular}{lrrrrrrrr}
\toprule
dataset & n_xai & zero_cam_count & n_valid & fap_05_mean & entropy_mean & confidence_mean & correct_rate & zero_cam_rate \\


## Table 8 - RQ6 CAM Agreement Across Datasets

In [10]:
if rq6_disc is None or len(rq6_disc) == 0:
    print("RQ6 CAM agreement summary not available.")
else:
    table8 = rq6_disc.copy()
    print(table8.round(4).to_string(index=False))
    print("\nLaTeX:")
    print(table8.round(4).to_latex(index=False, float_format="%.4f"))

         dataset     n  avg_jaccard_mean  min_jaccard_mean  max_jaccard_mean  correct_rate  confidence_mean
        HAM10000  9795            0.4651            0.1627            0.8912        0.8699           0.8963
        ISIC2020 32906            0.3813            0.0465            0.8737        0.9126           0.8936
         MILK10K 10014            0.4317            0.1075            0.8980        0.4198           0.8625
Malignant-Benign 34773            0.4501            0.1346            0.8974        0.6488           0.8655
             PH2   180            0.3868            0.0681            0.8826        0.5111           0.9277

LaTeX:
\begin{tabular}{lrrrrrr}
\toprule
dataset & n & avg_jaccard_mean & min_jaccard_mean & max_jaccard_mean & correct_rate & confidence_mean \\
\midrule
HAM10000 & 9795 & 0.4651 & 0.1627 & 0.8912 & 0.8699 & 0.8963 \\
ISIC2020 & 32906 & 0.3813 & 0.0465 & 0.8737 & 0.9126 & 0.8936 \\
MILK10K & 10014 & 0.4317 & 0.1075 & 0.8980 & 0.4198 & 0.8625 \\
Mal

## Final Results Summary and Discussion

In [11]:
print("=" * 90)
print("PAPER RESULTS SUMMARY")
print("=" * 90)

if rq1 is not None:
    rq1_mean = rq1.groupby("method")[["fap_05", "entropy"]].mean()
    best_fap = rq1_mean["fap_05"].idxmin()
    best_entropy = rq1_mean["entropy"].idxmin()
    print(f"RQ1: {best_fap} has the lowest FAP@0.5 ({rq1_mean.loc[best_fap, 'fap_05']:.4f}); {best_entropy} has the lowest entropy ({rq1_mean.loc[best_entropy, 'entropy']:.4f}).")

if rq2 is not None:
    rq2_mean = rq2.groupby("method")["faithfulness"].mean()
    best_faith = rq2_mean.idxmax()
    positive_rate = rq2.groupby("method")["faithfulness"].apply(lambda s: (s > 0).mean())
    print(f"RQ2: {best_faith} has the highest mean faithfulness ({rq2_mean.loc[best_faith]:.4f}) and positive-faithfulness rate ({positive_rate.loc[best_faith]:.2%}).")
    print("     Interpret this as relative faithfulness evidence, not proof of clinical trustworthiness.")

if rq3 is not None:
    rq3_summary = rq3.groupby(["model", "cam_method"])["fap_05"].mean().sort_values()
    top_model, top_cam = rq3_summary.index[0]
    print(f"RQ3: {top_model} + {top_cam} is the most focused observed combination (FAP@0.5={rq3_summary.iloc[0]:.4f}).")

if rq4 is not None:
    correct = rq4[rq4["correct"] == 1]["avg_jaccard"]
    incorrect = rq4[rq4["correct"] == 0]["avg_jaccard"]
    print(f"RQ4: CAM agreement is {correct.mean():.4f} for correct predictions and {incorrect.mean():.4f} for incorrect predictions.")
    print(f"     Incorrect n={len(incorrect)}; keep this as an exploratory risk-signal finding unless expanded.")

if rq6_perf is not None:
    ham = rq6_perf[rq6_perf["dataset"] == "HAM10000"]
    external = rq6_perf[rq6_perf["dataset"] != "HAM10000"].copy()
    if len(ham) and len(external):
        worst = external.sort_values("auc", na_position="last").iloc[0]
        overconf = external.sort_values("mean_confidence", ascending=False).iloc[0]
        print(f"RQ6: worst external AUC is {worst['dataset']} ({worst['auc']:.4f}); highest external confidence is {overconf['dataset']} ({overconf['mean_confidence']:.4f}).")
        print("     This supports a distribution-shift framing: performance, calibration, and explanation reliability must be reported together.")

if rq6_xai is not None and "zero_cam_count" in rq6_xai.columns:
    zero_total = int(rq6_xai["zero_cam_count"].sum())
    xai_total = int(rq6_xai["n_xai"].sum()) if "n_xai" in rq6_xai.columns else 0
    print(f"RQ6 XAI: zero-CAM failures = {zero_total}/{xai_total}. Report these as explanation failures, not as low-saliency successes.")

print("\nMy opinion for the paper:")
print("  The paper is strongest if it is positioned as a robustness study of CAM explanations under method choice, architecture choice, training stage, and external dataset shift.")
print("  The standout contribution is not just showing GradCAM overlays; it is quantifying when explanations remain focused/faithful and when they fail under distribution shift.")
print("  Avoid claiming clinical deployment readiness. The stronger, defensible claim is that external validation reveals overconfidence and explanation fragility that would be hidden by internal HAM10000 testing alone.")
print("  RQ4 and RQ5 are valuable but should be phrased as exploratory unless you expand their sample sizes.")
print("=" * 90)

PAPER RESULTS SUMMARY
RQ1: GradCAM has the lowest FAP@0.5 (0.0442); GradCAM has the lowest entropy (7.0243).
RQ2: GradCAM has the highest mean faithfulness (0.0828) and positive-faithfulness rate (62.00%).
     Interpret this as relative faithfulness evidence, not proof of clinical trustworthiness.
RQ3: resnet50 + GradCAM is the most focused observed combination (FAP@0.5=0.0458).
RQ4: CAM agreement is 0.2905 for correct predictions and 0.5354 for incorrect predictions.
     Incorrect n=25; keep this as an exploratory risk-signal finding unless expanded.
RQ6: worst external AUC is ISIC2020 (0.6348); highest external confidence is PH2 (0.9266).
     This supports a distribution-shift framing: performance, calibration, and explanation reliability must be reported together.
RQ6 XAI: zero-CAM failures = 82592/87618. Report these as explanation failures, not as low-saliency successes.

My opinion for the paper:
  The paper is strongest if it is positioned as a robustness study of CAM explana

## Useful Sources for the Discussion Section

- CLAIM 2024 medical imaging AI reporting checklist: https://pmc.ncbi.nlm.nih.gov/articles/PMC11304031/
- TRIPOD+AI 2024 prediction-model reporting guideline: https://www.bmj.com/content/385/bmj-2023-078378
- Adebayo et al., *Sanity Checks for Saliency Maps*, NeurIPS 2018: https://papers.nips.cc/paper/by-source-2018-5780
- Rong et al., *A Consistent and Efficient Evaluation Strategy for Attribution Methods*, ICML 2022 / ROAD: https://proceedings.mlr.press/v162/rong22a.html
- `pytorch-grad-cam` metrics reference, including confidence-change and ROAD metrics: https://github.com/jacobgil/pytorch-grad-cam

Use these sources to justify three discussion points: transparent medical-AI reporting, the need for external validation/calibration, and the risk of relying on visually plausible saliency maps without faithfulness or sanity checks.